In [ ]:
# !pip install undetected-chromedriver

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for undetected-chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47214 sha256=48cce2ec59392062772270913f959d317230a428912c0ddc1623b32d422288a8
  Stored in directory: c:\users\jeon\appdata\local\pip\cache\wheels\5c\b9\03\4b6e38f019d6170e8c25df2e1e362d7bdf9ff4012df2dc85c0
Successfully built undetected-chromedriver


# 세포라

In [5]:
import requests
import pandas as pd
import time
import json

def get_sephora_reviews(product_id, max_pages=5):
    all_reviews = []
    limit = 100  # API 효율을 위해 100으로 설정
    passkey = "calXm2DyQVjcCy9agq85vmTJv5ELuuBCF2sdg4BnJzJus"
    
    for page in range(max_pages):
        offset = page * limit
        url = "https://api.bazaarvoice.com/data/reviews.json"
        
        params = {
            "Filter": [f"contentlocale:en*", f"ProductId:{product_id}"],
            "Sort": "SubmissionTime:desc",
            "Limit": limit,
            "Offset": offset,
            "Include": "Products,Comments",
            "Stats": "Reviews",
            "passkey": passkey,
            "apiversion": "5.4",
            "Locale": "en_US"
        }
        
        try:
            response = requests.get(url, params=params)
            data = response.json()
            
            reviews = data.get('Results', [])
            
            if not reviews:
                print(f"\n✅ 모든 리뷰 수집 완료 (총 {len(all_reviews)}개)")
                break
                
            for rev in reviews:
                # 데이터 추출
                review_data = {
                    "ReviewId": rev.get("Id"),
                    "Author": rev.get("UserNickname"),
                    "Rating": rev.get("Rating"),
                    "Title": rev.get("Title"),
                    "ReviewText": rev.get("ReviewText"),
                    "SubmissionTime": rev.get("SubmissionTime"),
                    "IsRecommended": rev.get("IsRecommended"),
                    # 인센티브 여부 안전하게 추출
                    "Incentivized": rev.get("ContextDataValues", {}).get("IncentivizedReview", {}).get("ValueLabel", "N/A"),
                    "Helpfulness": rev.get("Helpfulness")
                }
                all_reviews.append(review_data)
            
            print(f"Page {page+1} 수집 중... (현재 {len(all_reviews)}개)", end='\r')
            time.sleep(0.5)
            
        except Exception as e:
            print(f"\n❌ 에러 발생: {e}")
            break
            
    # --- JSON 파일 저장 (리스트 형태 직접 저장) ---
    filename = f"sephora_test_crawling.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(all_reviews, f, ensure_ascii=False, indent=4)
    print(f"\n📂 '{filename}' 저장 완료!")
    
    return pd.DataFrame(all_reviews)

# --- 실행 부분 ---
PRODUCT_ID = "P514736"
df = get_sephora_reviews(PRODUCT_ID, max_pages=10)

# 데이터프레임 상위 5개 확인
if not df.empty:
    display(df.head())

Page 4 수집 중... (현재 323개)
✅ 모든 리뷰 수집 완료 (총 323개)

📂 'sephora_test_crawling.json' 저장 완료!


,ReviewId,Author,Rating,Title,ReviewText,SubmissionTime,IsRecommended,Incentivized,Helpfulness
0,380507344,danka81,1,Did absolutely nothing,The pads are super saturated by made zero diff...,2026-03-02T15:52:36.000+00:00,False,N/A,1.0
1,380477874,glamgirl303,3,Good glow!,"Good, but not great for actual brightening of ...",2026-03-02T02:10:50.000+00:00,True,No,1.0
2,378866022,marcelaalg,5,None,These are amazing!! Will continue to repurchas...,2026-02-10T22:11:39.000+00:00,True,No,1.0
3,378732803,COURTNEYEP,5,Love!,Love these! They are super soft and saturated!...,2026-02-09T07:18:33.000+00:00,True,No,NaN
4,378663889,RachelLynn232,5,Changed my skin!,Perfection! They have completely changed the c...,2026-02-08T05:04:10.000+00:00,True,No,NaN


# 라쿠텐

In [34]:
import requests
import json
import time
import pandas as pd

def get_rakuten_all_reviews(shop_id, item_id):
    all_reviews = []
    url = "https://web-gateway.rakuten.co.jp/review/itemshopreviewlist/get/v1"
    
    # ⚠️ 쿠키 인코딩 에러 방지 처리
    raw_cookie = r"_ra=1773145336745|9810936d-5cda-4630-8e49-c98379684815; Rp=dc4c4487b0d3c4dce21f7d6bdbf70329f03486c8; rcxGlobal=d45a9a4c-4afa-4759-a9ed-229577264fb8; bm_mi=02819FD1874A677A9FC3219D127ECD18~YAAQB9ojF3ykzaGcAQAAMAK01x85bJZA06vzzbG4DIKAXg7mopEgiKN5lzrV/0AErGfSM50oZ5QfIMqbrcROt7DYSkrYw58Cz7vwEO2XkjKMkHGsBRiolwhSHY6BjhqOg8UrHpjdOwbKwGHWbgXrhYVrXbpqnuYxOnS6Rd4tbRsuPIrmZrskES38smu59dSiBK0bGYfeffgKl/2sz7WT7sEbAOAom3GeHhE5vSnLp8Unxda00YTc54V1R0SccenoAvO/8tAjlqf4fcZMyVfgWjmkkXpBvobpj5hyrWagxWfFFgn06n/L/mwrI22pddxKke0Bln2zVz7w9QvyrImNfBfU7zu1YGnnS6HKKDs9fU21qukz3/qjiTlpM3yQIgMrYmSo2aL/dooM6uWJl/I7WeDfGzeHGvHpBy6zNQ==~1; bm_sv=F3073B8724961BB7DCBE5A9516653181~YAAQbIj+eWOAwsucAQAA7Di01x/kkLOsyzmShVkJPcJf1O0TkzKdl7ZL7qgaen5E33IuCROVCPlrVdQr767lxL25yZRnHKhQTLV74swQf/WGDo3WVE2ehk0W7uv6G6PUyNE2N5dQyBl/0tnQF1TIL1GoIY1QVyysRmPhNLu9lndOUVbr8CgHlwjK+8cZfXcWG/rrLNFiEsHCtxhiQUijgpAP4TGE1raLDQ8pAaWOcclxAHnAjyBcQ2TkH+CwcSLNeCnH~1; ak_bmsc=6DA2C442401E92959215D4FC32333198~000000000000000000000000000000~YAAQB9ojF5AVzqGcAQAA0cu01x9fj3L5e+d9RAJ1hoLlH3K8NWtbYVM99UU1cPhBjs0l+RT4L93zSS2KCBZ2cF79c9IkAVFdcgGHbxP50Uf8nlO5I6u+fsByVK6zS4EhUNSCSizuhMPBdf0lHio/+lSN+8w0fTRgZfWwPNbWNRlfyg8SAriAihvo7Qf2txj9kJZGnaHPqgkCZXs11UwhehZ5Ak3MzYtTcojSc91bzqp+zzmmEbyhuYzqW5FWGr6d5k6LF157rv/BKCBjyW2dtk7TbRgJKgQGZz/gPYD8n8mDMroN36ptPxuByW5GOOwTwoCs8tDnUOjgOtm8M7mmdRNKlpJsUSJoOylT7fkQoeVPZkHsLG1j5QOPG60+P7zXsIeB4pddGO/PKsDF8HPAgLpOVb2Dgqz56kyCR4vfPNXZLawqlDXl8+8HXh2AL5KJmQIZMy0XvT/jsyXGkSBk6KO8l/jsyQp1AG570i/fWulONUZGzkQKF86JqOOaoDS7dUrDgp1mqvDR4lwtHSs=; krt_rewrite_uid=f8b15f33-cc3e-45c0-b1be-9a29a4cff1d2; Re=31.1.5.0.0.216348.3-31.1.5.0.0.216348.3; rat_v=05252dcb01d4e6863512b1a3e769b00e711919a"
    safe_cookie = raw_cookie.encode('utf-8').decode('latin-1', 'ignore')

    headers = {
        "authkey": "isrlPcMjUuXCVBUTVh91ZcHEfoI45CmPR",
        "content-type": "application/json; charset=UTF-8",
        "accept": "application/json, text/plain, */*",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
        "cookie": safe_cookie,
        "referer": "https://review.rakuten.co.jp/"
    }

    page = 1
    has_next = True

    print(f"🚀 상품 {item_id} 모든 리뷰 수집 시작 (전체 약 2,200여 건)...")

    while has_next:
        # ✅ 중첩 구조 Payload 반영
        payload = {
            "common": {
                "params": {"device": "pc"},
                "include": ["itemReviewList"]
            },
            "features": {
                "itemReviewList": {
                    "params": {
                        "shopId": int(shop_id),
                        "itemId": int(item_id),
                        "sort": "",
                        "page": str(page),
                        "hits": 30,
                        "filter": {"rating": "", "mediaOnly": "false", "ageRange": "", "sex": ""},
                        "includePickupReview": True
                    }
                }
            }
        }

        try:
            response = requests.post(url, headers=headers, json=payload)
            
            # 200 또는 207 코드가 오면 정상
            if response.status_code not in [200, 207]:
                print(f"\n❌ {page}p 중단 (코드:{response.status_code})")
                break

            data = response.json()
            
            # 파싱 경로: body -> itemReviewList -> data
            res_body = data.get("body", {}).get("itemReviewList", {})
            res_data = res_body.get("data", {})
            reviews = res_data.get("reviews", [])
            
            if not reviews:
                print(f"\n✅ {page}페이지 결과 없음. 수집 완료.")
                break
                
            for rev in reviews:
                all_reviews.append({
                    "Page": page,
                    "Nickname": rev.get("nickname"),
                    "Rating": rev.get("rating"),
                    "Body": rev.get("body"),
                    "PostDate": rev.get("postDate"),
                    "Age": f"{rev.get('ageRange', '')}{rev.get('ageSuffix', '')}",
                    "Sex": rev.get("sex"),
                    "Sku": rev.get("skuInfo")
                })
            
            # ✅ 다음 페이지 존재 여부 확인
            has_next = res_data.get("hasNextPage", False)
            
            print(f"🔄 {page}페이지 완료... (누적 {len(all_reviews)}개)", end='\r')
            
            page += 1
            # 라쿠텐 보안을 고려하여 1.5~2초 간격 권장
            time.sleep(1.8)

        except Exception as e:
            print(f"\n❌ 에러 발생: {e}")
            break

    # 최종 저장
    if all_reviews:
        filename = f"rakuten_{shop_id}_{item_id}_ALL.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(all_reviews, f, ensure_ascii=False, indent=4)
        print(f"\n📂 총 {len(all_reviews)}개 데이터 저장 완료! ('{filename}')")
    
    return pd.DataFrame(all_reviews)

# --- 실행 ---
SHOP_ID = "371043"
ITEM_ID = "10000027"
df = get_rakuten_all_reviews(SHOP_ID, ITEM_ID)

🚀 상품 10000027 모든 리뷰 수집 시작 (전체 약 2,200여 건)...
🔄 76페이지 완료... (누적 2280개)
📂 총 2280개 데이터 저장 완료! ('rakuten_371043_10000027_ALL.json')


# ulta

In [38]:
import requests
import json
import time
import pandas as pd

def get_ulta_reviews_auto_stop(product_id):
    all_reviews = []
    page_size = 5  # 이미지와 동일하게 5개씩 호출
    paging_from = 0
    apikey = "daa0f241-c242-4483-afb7-4449942d1a2b"
    
    print(f"🚀 {product_id} 상품 리뷰 수집을 시작합니다. (데이터가 없을 때까지 무한 반복)")

    while True:
        # 매번 paging_from 값이 0, 5, 10...으로 업데이트됨
        url = f"https://display.powerreviews.com/m/6406/l/en_US/product/{product_id}/reviews"
        
        params = {
            "paging.from": paging_from,
            "paging.size": page_size,
            "sort": "Newest",
            "image_only": "false",
            "page_locale": "en_US",
            "_noconfig": "true",
            "apikey": apikey
        }
        
        try:
            response = requests.get(url, params=params)
            
            if response.status_code != 200:
                print(f"\n❌ 서버 응답 에러 (코드: {response.status_code})")
                break
                
            data = response.json()
            results = data.get('results', [])
            
            # results 리스트가 비어있거나 내부의 reviews가 없으면 즉시 종료
            if not results or not results[0].get('reviews'):
                print(f"\n✅ 수집 완료: 더 이상 가져올 데이터가 없습니다.")
                break
            
            reviews = results[0].get('reviews', [])
            
            for rev in reviews:
                details = rev.get("details", {})
                metrics = rev.get("metrics", {})
                
                all_reviews.append({
                    "ReviewId": rev.get("review_id"),
                    "Author": details.get("nickname"),
                    "Rating": metrics.get("rating"),
                    "Headline": details.get("headline"),
                    "Comments": details.get("comments"),
                    "Location": details.get("location"),
                    "CreatedDate": details.get("created_date"),
                    "HelpfulVotes": metrics.get("helpful_votes")
                })
            
            print(f"🔄 수집 중... (현재 누적: {len(all_reviews)}개)", end='\r')
            
            # 다음 페이지로 이동
            paging_from += page_size
            time.sleep(1) # 차단 방지용

        except Exception as e:
            print(f"\n❌ 에러 발생: {e}")
            break
            
    # --- 최종 JSON 저장 ---
    output_filename = "ulta_test_crawling.json"
    if all_reviews:
        with open(output_filename, 'w', encoding='utf-8') as f:
            json.dump(all_reviews, f, ensure_ascii=False, indent=4)
        print(f"\n📂 수집 성공! '{output_filename}' 파일에 {len(all_reviews)}개 저장됨.")
    else:
        print("\n⚠️ 저장할 데이터가 없습니다.")
    
    return pd.DataFrame(all_reviews)

# --- 실행 ---
PRODUCT_ID = "pimprod2049597"
df = get_ulta_reviews_auto_stop(PRODUCT_ID)

🚀 pimprod2049597 상품 리뷰 수집을 시작합니다. (데이터가 없을 때까지 무한 반복)
🔄 수집 중... (현재 누적: 218개)
✅ 수집 완료: 더 이상 가져올 데이터가 없습니다.

📂 수집 성공! 'ulta_test_crawling.json' 파일에 218개 저장됨.


In [45]:
import requests
from bs4 import BeautifulSoup
import json
import time
import pandas as pd

def get_qoo10_emergency_fix(gd_no):
    all_reviews = []
    page = 1
    base_url = "https://www.qoo10.jp/gmkt.inc/Goods/GoodsReviewAjaxAppend.aspx"
    
    # 서버 차단을 피하기 위해 브라우저와 거의 동일한 헤더 설정
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
        "Accept": "text/html, */*",
        "Accept-Language": "ja,en-US;q=0.9,en;q=0.8,ko;q=0.7",
        "Referer": f"https://www.qoo10.jp/item/x/{gd_no}", # 실제 상품 페이지 주소를 참조로 넣음
        "X-Requested-With": "XMLHttpRequest" # Ajax 요청임을 명시
    }

    print(f"🚀 Qoo10 상품 {gd_no} 수집 재시도 중...")

    while True:
        params = {
            "gd_no": gd_no,
            "group_code": "2",
            "page_no": page,
            "page_size": "10",
            "sort_type": "P",
            "contents_cnt": "0",
            "___cache_expire___": int(time.time() * 1000)
        }
        
        try:
            response = requests.get(base_url, params=params, headers=headers)
            
            # 응답 상태 확인
            if response.status_code != 200:
                print(f"\n❌ 서버 응답 에러 (코드: {response.status_code})")
                break
                
            html_content = response.text.strip()
            if not html_content:
                print(f"\n✅ 수집 완료: 더 이상 읽을 HTML 데이터가 없습니다.")
                break
            
            soup = BeautifulSoup(html_content, 'html.parser')
            
            # 💡 수정된 포인트: li 태그를 찾되, 클래스나 구조에 상관없이 
            # 실제 리뷰 텍스트가 들어있는 영역을 먼저 탐색합니다.
            review_items = soup.find_all('li', recursive=False)
            
            if not review_items:
                # 가끔 상위 태그 없이 li만 올 때가 있으므로 다시 확인
                review_items = soup.select('li')
                
            if not review_items:
                break
                
            for item in review_items:
                # 1. 리뷰 본문 추출 (review_txt 클래스가 핵심)
                txt_tag = item.select_one('.review_txt')
                if not txt_tag: continue # 본문이 없으면 리뷰가 아닐 확률이 높음
                
                content = txt_tag.get_text(strip=True)
                
                # 2. 평점 (score 클래스 또는 스타일 너비)
                score_tag = item.select_one('.score')
                rating = score_tag.get_text(strip=True) if score_tag else "5" # 기본값 5
                
                # 3. 작성자 및 메타정보
                user_info = item.select_one('.review_user_info')
                user_meta = user_info.get_text(" | ", strip=True) if user_info else ""
                
                # 4. 피부 타입/고민
                type_tag = item.select_one('.review_user_type')
                skin_info = type_tag.get_text(strip=True) if type_tag else ""

                all_reviews.append({
                    "Page": page,
                    "Rating": rating,
                    "Review": content,
                    "UserInfo": user_meta,
                    "SkinType": skin_info
                })
            
            print(f"🔄 {page}페이지 수집 중... (누적: {len(all_reviews)}개)", end='\r')
            
            page += 1
            # 💡 차단을 방지하기 위해 지연 시간을 약간 늘립니다.
            time.sleep(1)

        except Exception as e:
            print(f"\n❌ 실행 에러: {e}")
            break
            
    # --- 저장 ---
    output_name = "qoo10_test_crawling.json"
    if all_reviews:
        with open(output_name, 'w', encoding='utf-8') as f:
            json.dump(all_reviews, f, ensure_ascii=False, indent=4)
        print(f"\n📂 수집 성공! 총 {len(all_reviews)}개 저장 완료.")
    
    return pd.DataFrame(all_reviews)

# --- 실행 ---
df = get_qoo10_emergency_fix("501192305")

🚀 Qoo10 상품 501192305 수집 재시도 중...
🔄 193페이지 수집 중... (누적: 1923개)
✅ 수집 완료: 더 이상 읽을 HTML 데이터가 없습니다.

📂 수집 성공! 총 1923개 저장 완료.
